In [1]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt

    
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

#
import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np


#
import scipy
import os
import time

#
import pickle 


#
from calibration.CalibrationTools import (CalibrationTools, get_binary_std_map, get_rois_stardist2d, get_img_std,
                                          save_calibration_data, get_footprints_from_suite2p)


#from drift.drift import (make_template, compute_drift_multi_frames, correct_drift, 
#                         correct_drift_single_frame, template_generation, 
 #                        plot_mean_vs_template, make_motion_template_and_correct_data)
from utils.utils import smooth_ca_time_series, compute_dff0

from utils.calcium import calcium


Operating system:  Linux


Autosaving every 180 seconds


In [2]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################

#
fname = '/media/cat/4TB/donato/andres/DON011733/day0/calibration/Image_001_001.raw'
fname = '/media/cat/4TB/donato/DON-010798/day0/calibration/Image_001_001.raw'


# 
bmi_c = CalibrationTools(fname)
bmi_c.calcium = calcium
#
bmi_c.smooth_ca_time_series = smooth_ca_time_series
bmi_c.compute_dff0 = compute_dff0

#
bmi_c.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is fast in other implemenations

#
bmi_c.std_map = bmi_c.data[:1000].mean(0)

#
# load suite2p footprints
bmi_c = get_footprints_from_suite2p(bmi_c)

# 
print ("DONE...")

memmap :  (27000, 512, 512)
  Fluorescence data loading information
         sample rate:  30 hz
         self.F (fluorescence):  (536, 27000)
         self.Fneu (neuropile):  (536, 27000)
         self.iscell (Suite2p cell classifier output):  (584, 2)
              of which number of good cells:  (536,)
         self.spks (deconnvoved spikes):  (536, 27000)
         self.stat (footprints structure):  (536,)
         mean std over all cells :  65.0
# of footprints;  536
DONE...


In [3]:
###########################################################
################# AUTOMATE CELL SELECTION: 1/2 ############
###########################################################

# OPTION #1: AUTOMATICALLY SELECT CELSS
# 

# initialize calcium object and load suite2p data
bmi_c.c = calcium.Calcium()
bmi_c.c.verbose=False
bmi_c.c.data_dir = os.path.join(os.path.split(fname)[0],'suite2p','plane0')
bmi_c.c.recompute_binarization = False
bmi_c.c.min_thresh_std_onphase = 10  # use a high conservative zscore to detect really the biggest bursts
bmi_c.c.min_thresh_std_upphase = 10  # set the minimum thershold for uppohase detection; default: 2.5
bmi_c.c.detrend_model_order = 1  # 1-5 polynomial fit
bmi_c.c.min_width_event_onphase = bmi_c.c.sample_rate // 2  # set minimum withd of an onphase event; default: 0.5 seconds
bmi_c.c.min_width_event_upphase = bmi_c.c.sample_rate // 2  # set minimum width of upphase event; default: 0.25 seconds
bmi_c.c.verbose=False

bmi_c.c.run_binarization_bmi()
bmi_c.c.load_binarization()

# also compute # of bursts per cell
bmi_c.c.compute_bursts_using_upphase=False
bmi_c.c.compute_n_bursts_per_cell()

# visualize bursts
cell_id = 35
scale=1000

#
print ("Cell has # bursts: ", bmi_c.c.n_bursts[cell_id])

#

#########################################################
############### AUTO GENERATE CELL IDS ##################
#########################################################
# generate SNRS
bmi_c.compute_roi_traces_f0_no_reorder()  # this function also coputes the snr / f0s of the cells



  Fluorescence data loading information
         sample rate:  30 hz
         self.F (fluorescence):  (536, 27000)
         self.Fneu (neuropile):  (536, 27000)
         self.iscell (Suite2p cell classifier output):  (584, 2)
              of which number of good cells:  (536,)
         self.spks (deconnvoved spikes):  (536, 27000)
         self.stat (footprints structure):  (536,)
         mean std over all cells :  65.0
   todo: print binarization defaults...
(536, 27000)
Cell has # bursts:  8


computing roi traces for SNR indexing: 100%|██████████████████| 2700/2700 [00:09<00:00, 291.68it/s]


In [9]:
##############################################################
############### AUTO CELL SELECTION: 2/2 #####################
##############################################################
#
bmi_c.ensemble_burst_rate = 0.5   # no. of bursts per minute
bmi_c.max_distance_ensemble_cells = 50
bmi_c.top_cells = 75              # top N cells to choose from
bmi_c.low_bound = 0.5
bmi_c.upper_bound = 3

#
bmi_c.min_snr = 1
bmi_c.n_iter = 500               # # of times to run ; DON'T CHANGE THIS
bmi_c.auto_generate_ensembles()

# GO TO VISUALIZE CELL 3 CELLS DOWN

ideal burst rate:  0.5  per min.
 total # bursts:  7
ensembel1 distance:  16.397457107070693
E1 cells:  46 14
ensembel2 distance:  29.952856476655274
E2 cells:  36 16


In [10]:
# #########################################################
# # ########### REORDER CELLS BY SNR OR F0 ##################
# # #########################################################
# order_type = 'snr'  # 'snr' or 'f0'
#bmi_c.compute_roi_traces_f0_no_reorder()  # this function also coputes the snr / f0s of the cells

# save ensemble rois
bmi_c.ensemble1 = [bmi_c.E1_1, bmi_c.E1_2]
bmi_c.ensemble2 = [bmi_c.E2_1, bmi_c.E2_2]
bmi_c.both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
print ("all cells:", bmi_c.both)

#
# bmi_c.show_traces_ids(bmi_c.both)
# # ######################################################################
# # ########### RECOMPUTE TRACES WITH SINGLE FRAME PRECISION #############
# # ######################################################################
bmi_c.trace_subsample = 10        # Subsample the time series to go faster;

# visualize traces
#bmi_c.compute_traces2(std_map, both)
bmi_c.vmin = 0
bmi_c.vmax = 1000
bmi_c.compute_traces_ensembles2(bmi_c.std_map)
plt.xlim(0,512)
plt.ylim(0,512)
print ("DONE...")


all cells: [46 14 36 16]


100%|██████████████████████████████████████████████████████| 27000/27000 [00:03<00:00, 7001.37it/s]


Contour:  [336 306]
cell ids:  [46 14 36 16]
DONE...


In [11]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################
# 
bmi_c.sample_rate = 30
bmi_c.post_reward_lockout = 3   # reward lockout in seconds
bmi_c.post_missed_reward_lockout = 10
bmi_c.trial_time = 30
bmi_c.post_reward_lockout_baseline_min = 0.3
                                 # TODO: in future load/save this to disk so that BMI 
                                 #   - doesn't use differetn lockout than calibration step
bmi_c.balance_ensemble_rewards_flag = False   #this makes sure that both ensembles elicit a similar number of random rewards
bmi_c.rois_smooth_window = 5     # of frames to use to smooth the realtime signal
bmi_c.smooth_diff_function_flag = True    # use a kernel window to smooth current value

#
# find 30% reward threshold
bmi_c.reward_rate = 0.25#*0.85

#gr.find_reward_thresholds_low_and_high()
#bmi_c.high = 2
bmi_c.find_reward_thresholds_high_realtime()  # this only rewards when sound passes specific level

#
print ("thresholds: ", bmi_c.high)

############################################## 
bmi_c.plot_rewarded_ensembles()


# do not change this
bmi_c.reward_rate_scaling_factor = 1.0

#
bmi_c.high = bmi_c.high*bmi_c.reward_rate_scaling_factor
print ("bmi_c.high: scaled by: ", bmi_c.reward_rate_scaling_factor, ", final value:  ", bmi_c.high)


nsec recording:  900 max # of random rewards (i.e. every 30sec)  30
 @0.25% reward rate:  7
 high guess:  3.9634391405327714
updated rewards #:  0 3.9634391405327714
updated rewards #:  0 3.7652671835061327
updated rewards #:  0 3.5770038243308258
updated rewards #:  0 3.3981536331142843
updated rewards #:  0 3.22824595145857
updated rewards #:  1 3.066833653885641
updated rewards #:  1 2.913491971191359
updated rewards #:  1 2.7678173726317907
updated rewards #:  1 2.629426504000201
updated rewards #:  4 2.497955178800191
updated rewards #:  5 2.373057419860181
updated rewards #:  5 2.254404548867172
updated rewards #:  5 2.141684321423813
updated rewards #:  5 2.0346001053526224
updated rewards #:  5 1.932870100084991
updated rewards #:  5 1.8362265950807415
updated rewards #:  6 1.7444152653267044
updated rewards #:  6 1.6571945020603691
updated rewards #:  7 1.5743347769573506
updated rewards #:  7 1.5743347769573506
thresholds:  1.5743347769573506
 PROCESSING...
bmi_c.high: scaled

In [8]:
#############################################
########### SAVE DATA #######################
#############################################

#    
text = "day0"
bmi_c = save_calibration_data(bmi_c,text)  

contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is

/home/cat/.conda/envs/bmi/lib/python3.8/site-packages/numpy/lib/npyio.py:716: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  val = np.asanyarray(val)


 couldn't save bmi_c.object .... TO FIX!
Done...
